In [1]:
import pandas as pd
import requests 
import json
import os
import networkx as nx

import matplotlib.pyplot as plt
#from wordcloud import WordCloud

from collections import defaultdict
import spacy
#import Functions as fn
#import seaborn as sns
import numpy as np  

import re

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
nlp = spacy.load("en_core_sci_sm") 

/opt/conda/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [4]:
pwd

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [5]:
home_dir = '/home/eidf128/eidf128/shared/export/juliana/export/juliana'

### Functions

##### Clean

In [6]:
def remove_stop_words(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    filtered = [w for w in tokens if w.lower() not in stop_words and w.isalpha()]

    return filtered

In [7]:
def remove_special_characters(text):
    """
    This function removes special characters from the text.
    :param text: text
    :return: text without special characters
    """
    cleaned_string = re.sub(r'[?|$|.|!|@|#|%|^|&|*|(|)|-|_|+|=|;|:|,|<|>|/|{|}|[|]|~|`|\'|\"|\\]',r'', text)

    return cleaned_string


In [8]:
def clean_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = remove_special_characters(str(text)) or ""
    
    return remove_stop_words(text)

##### Counts

In [9]:
# It counts the number of words in a phrase
def count_words(phrase):
    """
    This function counts the number of words in a phrase.
    :param phrase: text

    :return: the number of words in the phrase
    """
    lenght = 0
    if len(phrase) > 1 and phrase.isspace() == True:
        lenght = 1
    
    else:
        words = phrase.split()
        lenght = len(words)
    
    return lenght

In [10]:
# It counts the number of characters in a phrase
def count_characters(phrase):
    """
    This function counts the number of characters in a phrase.
    :param phrase: text

    :return: the number of characters in the phrase
    """
    lenght = 0
    if len(phrase) < 1:
        lenght = 0
    else:
        lenght = len(phrase)
        
    return lenght

#### NLP

In [11]:
# It lemmizes a text
def lemmatizer(phrase):
    """
    This function lemmatizes a text.
    :param phrase: text

    :return: the lemmatized text
    """
    doc = nlp(phrase)
    lemmatized_tokens = [token.lemma_ for token in doc]
    lemmatized_text = ' '.join(lemmatized_tokens)
    
    return lemmatized_text

In [12]:
# It extracts the entities from a text
def entities_recognition(phrase):
    """
    This function extracts the entities from a text.
    :param phrase: text
    :return: the entities extracted from the text
    """
    entities = []
    doc_phrase = nlp(phrase)
    entities = list(doc_phrase.ents)
    
    return entities

##### Organization

In [13]:
def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA

In [14]:
def join_unique_non_null(s, sep=" | "):
    s = s.dropna().astype(str)
    s = s[s.str.strip().ne("")]           # drop empty strings
    s = pd.unique(s)                      # keep unique (preserve order)
    return sep.join(s) if len(s) else pd.NA

### Reading the data and creating df

In [15]:
data_dir = '/home/eidf128/eidf128/shared/export/juliana/export/juliana/items'

In [16]:
os.chdir(data_dir)

In [17]:
json_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]
json_files

['49dbc9e1-eeab-411b-8510-080ceb5dccfa_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 'ecfd3009-d585-4b1a-9137-6b7460379b94_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 '619bb04f-94c4-4eb8-9511-b2dfeaa3412a_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 'e232d2db-1621-47df-bb6d-ea69187de44f_in_3b8e9ad5-7d72-430c-aed5-a73376ba2bf8.json',
 '9bf808da-4a87-4bfc-a885-4f53bf7531fc_in_300bc74f-b2ca-4cdb-94f8-757e57d8f753.json',
 '50e06d54-ef79-4eda-ba70-53af5f601281_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 '4e3448ea-fc40-4b5c-94f4-a429349bbc27_in_456eccd4-54f2-48f2-afb0-c4a02936452c.json',
 'c453e153-7b18-4b3d-ae05-69ef3210e4c7_in_9053e53c-3f3e-443a-940a-bf374489f89c.json',
 '2f73dbf5-7b4a-4f18-a29c-20a046031c39_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 '35241ba3-5bd0-4266-966f-fbbdedf2d53b_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 '687d6803-5d6e-447d-b9ed-d9f71d4a3d26_in_78be837d-99a1-4da9-813d-fa139b2853f2.json',
 'f084fe67-dfb5-4569-b9a1-759ef2ddbdc3_in_51c495e9-9c4

In [18]:
dfs_datashare = []

In [19]:
for json_file in json_files:
    json_1 = open(json_file)
    json_dicc = json.load(json_1)

    file_name = os.path.basename(json_file)
    
    result = defaultdict(list)

    for item in json_dicc:
        key = item['key']
        value = item['value']
        result[key].append(value)

    result_dict = dict(result)

    df = pd.json_normalize(result_dict)
    name = file_name.split('_')[0]
    collection = file_name.split('_')[2]
    df['file_name'] = file_name
    df['collection'] = collection.split('.')[0]
    df['id_in_file'] = name

    dfs_datashare.append(df)

df_datashare_collapse = pd.concat(dfs_datashare)

In [20]:
df_datashare_collapse.columns

Index(['dc.contributor', 'dc.contributor.other', 'dc.coverage.spatial',
       'dc.coverage.temporal', 'dc.creator', 'dc.date.accessioned',
       'dc.date.available', 'dc.identifier.citation', 'dc.identifier.uri',
       'dc.description.abstract', 'dc.language.iso', 'dc.publisher',
       'dc.relation.isreferencedby', 'dc.rights', 'dc.subject',
       'dc.subject.classification', 'dc.title', 'dc.type', 'file_name',
       'collection', 'id_in_file', 'dc.relation.isversionof',
       'dc.description.tableofcontents', 'dc.source', 'dc.title.alternative',
       'dc.relation.isreplacedby', 'dc.date.updated', 'dc.date.embargo',
       'dc.relation.replaces', 'dc.source.uri', 'dc.date.issued',
       'dcterms.subject', 'dcterms.isReferencedBy', 'dc.contributor.advisor',
       'dc.relation.hasversion', 'dc.contributor.author', 'dc.description',
       'dc.description.sponsorship', 'dc.relation.ispartofseries',
       'ds.withdrawn.showtombstone', 'dc.relation.isbasedon', 'dcterms.rights',


### Cleaning

In [21]:
# Replace None with nothing
df_datashare_collapse.replace(to_replace=[None], value="", inplace=True)

,dc.contributor,dc.contributor.other,dc.coverage.spatial,dc.coverage.temporal,dc.creator,dc.date.accessioned,dc.date.available,dc.identifier.citation,dc.identifier.uri,dc.description.abstract,...,dc.contributor.author,dc.description,dc.description.sponsorship,dc.relation.ispartofseries,ds.withdrawn.showtombstone,dc.relation.isbasedon,dcterms.rights,dc.relation.ispartof,dcterms.isReplacedBy,dcterms.publisher
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[Wolverhampton City, United Kingdom]",[start=1987; end=1987; scheme=W3C-DTF],"[Glendinning, Miles]",[2023-05-17T16:00:22Z],[2023-05-17T16:00:22Z],"[Glendinning, Miles. (2023). Tower Blocks UK: ...","[https://hdl.handle.net/10283/8181, https://do...",[Multi-storey block details: two 8-storey bloc...,...,,,,,,,,,,
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[Enfield London, United Kingdom]",[start=1988; end=1988; scheme=W3C-DTF],"[Glendinning, Miles]",[2023-05-17T13:55:10Z],[2023-05-17T13:55:10Z],"[Glendinning, Miles. (2023). Tower Blocks UK: ...","[https://hdl.handle.net/10283/5559, https://do...",[Multi-storey block details: four 12-storey bl...,...,,,,,,,,,,
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[AB11 8TW; AB11 8TX; AB11 8TY, Aberdeen, UK, U...",[start=1983; end=1983; scheme=W3C-DTF],"[Glendinning, Miles]",[2019-11-14T09:55:05Z],[2019-11-14T09:55:05Z],"[Glendinning, Miles. (2019). SC_1141943.png, B...","[https://hdl.handle.net/10283/3445, https://do...",[Multi-storey block details: three 14-storey b...,...,,,,,,,,,,
0,"[Kim, Jinho]",[University of Edinburgh],,,"[Kim, Jinho]",[2017-06-14T11:00:41Z],[2017-06-14T11:00:41Z],"[Kim, Jinho. (2017). Source codes of the netwo...","[https://hdl.handle.net/10283/2737, https://do...",[This data set contains the source codes for t...,...,,,,,,,,,,
0,"[Anjos, Miguel]",[University of Edinburgh],,,"[Burkard, RE, Çela, E, Karisch, SE, Rendl, F, ...",[2022-03-31T16:59:08Z],[2022-03-31T16:59:08Z],"[Burkard, RE; Çela, E; Karisch, SE; Rendl, F; ...","[https://hdl.handle.net/10283/4390, https://do...",[The Quadratic Assignment Problem (QAP) has re...,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,"[Nadal, Cathe Desiree]",[Self-funded],"[Cavite, Cavite, Philippines, PH, PHILIPPINES]",,"[Nadal, Cathe Desire, Del Rosario, Tovie Clart...",[2022-10-11T14:40:23Z],[2022-10-11T14:40:23Z],"[Nadal, Cathe Desire; Del Rosario, Tovie Clart...","[https://hdl.handle.net/10283/4758, https://do...","[""CAPTURING THE DYNAMIC CHARACTER OF THE PHILI...",...,,,,,,,,,,
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[Sefton, United Kingdom]",[start=1987; end=1987; scheme=W3C-DTF],"[Glendinning, Miles]",[2023-05-17T15:43:57Z],[2023-05-17T15:43:57Z],"[Glendinning, Miles. (2023). Tower Blocks UK: ...","[https://hdl.handle.net/10283/7742, https://do...",[Multi-storey block details: one 16-storey blo...,...,,,,,,,,,,
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[Camden London, United Kingdom]",[start=1988; end=1988; scheme=W3C-DTF],"[Glendinning, Miles]",[2023-05-17T13:34:44Z],[2023-05-17T13:34:44Z],"[Glendinning, Miles. (2023). Tower Blocks UK: ...","[https://hdl.handle.net/10283/5355, https://do...",[Multi-storey block details: CS 1: eleven 6-st...,...,,,,,,,,,,
0,"[Glendinning, Miles, Breen, Kat]",[Heritage Lottery Fund],"[Salford City, United Kingdom]",[start=1987; end=1987; scheme=W3C-DTF],"[Glendinning, Miles]",[2023-05-17T15:41:06Z],[2023-05-17T15:41:06Z],"[Glendinning, Miles. (2023). Tower Blocks UK: ...","[https://hdl.handle.net/10283/7619, https://do...",[Multi-storey block details: one 20-storey blo...,...,,,,,,,,,,


In [22]:
# Everything to lowercase
obj_cols = df_datashare_collapse.select_dtypes(include = ["object", "string"]).columns #Select columns that have strings on it
df_datashare_collapse[obj_cols] = df_datashare_collapse[obj_cols].astype("string").apply(lambda s: s.str.lower())

### Stats

In [23]:
df_datashare_collapse.columns

Index(['dc.contributor', 'dc.contributor.other', 'dc.coverage.spatial',
       'dc.coverage.temporal', 'dc.creator', 'dc.date.accessioned',
       'dc.date.available', 'dc.identifier.citation', 'dc.identifier.uri',
       'dc.description.abstract', 'dc.language.iso', 'dc.publisher',
       'dc.relation.isreferencedby', 'dc.rights', 'dc.subject',
       'dc.subject.classification', 'dc.title', 'dc.type', 'file_name',
       'collection', 'id_in_file', 'dc.relation.isversionof',
       'dc.description.tableofcontents', 'dc.source', 'dc.title.alternative',
       'dc.relation.isreplacedby', 'dc.date.updated', 'dc.date.embargo',
       'dc.relation.replaces', 'dc.source.uri', 'dc.date.issued',
       'dcterms.subject', 'dcterms.isReferencedBy', 'dc.contributor.advisor',
       'dc.relation.hasversion', 'dc.contributor.author', 'dc.description',
       'dc.description.sponsorship', 'dc.relation.ispartofseries',
       'ds.withdrawn.showtombstone', 'dc.relation.isbasedon', 'dcterms.rights',


In [24]:
df_datashare_collapse["dc.description.abstract_clean"] = pd.Series(index = df_datashare_collapse.index, dtype="object")

In [25]:
print(df_datashare_collapse.shape)

(7754, 46)


In [26]:
df_datashare_collapse = df_datashare_collapse.reset_index(drop=True)

In [27]:
print(df_datashare_collapse.index)
print(df_datashare_collapse.index.is_unique)
print(df_datashare_collapse.index.value_counts().head())


RangeIndex(start=0, stop=7754, step=1)
True
0    1
1    1
2    1
3    1
4    1
Name: count, dtype: int64


In [28]:
type(df_datashare_collapse)

pandas.DataFrame

In [29]:
df_datashare_collapse

,dc.contributor,dc.contributor.other,dc.coverage.spatial,dc.coverage.temporal,dc.creator,dc.date.accessioned,dc.date.available,dc.identifier.citation,dc.identifier.uri,dc.description.abstract,...,dc.description,dc.description.sponsorship,dc.relation.ispartofseries,ds.withdrawn.showtombstone,dc.relation.isbasedon,dcterms.rights,dc.relation.ispartof,dcterms.isReplacedBy,dcterms.publisher,dc.description.abstract_clean
0,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['wolverhampton city', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t16:00:22z'],['2023-05-17t16:00:22z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/8181', 'https:/...","[""multi-storey block details: two 8-storey blo...",...,,,,,,,,,,NaN
1,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['enfield london', 'united kingdom']",['start=1988; end=1988; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t13:55:10z'],['2023-05-17t13:55:10z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/5559', 'https:/...","[""multi-storey block details: four 12-storey b...",...,,,,,,,,,,NaN
2,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['ab11 8tw; ab11 8tx; ab11 8ty', 'aberdeen', '...",['start=1983; end=1983; scheme=w3c-dtf'],"['glendinning, miles']",['2019-11-14t09:55:05z'],['2019-11-14t09:55:05z'],"['glendinning, miles. (2019). sc_1141943.png, ...","['https://hdl.handle.net/10283/3445', 'https:/...","[""multi-storey block details: three 14-storey ...",...,,,,,,,,,,NaN
3,"['kim, jinho']",['university of edinburgh'],,,"['kim, jinho']",['2017-06-14t11:00:41z'],['2017-06-14t11:00:41z'],"['kim, jinho. (2017). source codes of the netw...","['https://hdl.handle.net/10283/2737', 'https:/...",['this data set contains the source codes for ...,...,,,,,,,,,,NaN
4,"['anjos, miguel']",['university of edinburgh'],,,"['burkard, re', 'çela, e', 'karisch, se', 'ren...",['2022-03-31t16:59:08z'],['2022-03-31t16:59:08z'],"['burkard, re; çela, e; karisch, se; rendl, f;...","['https://hdl.handle.net/10283/4390', 'https:/...",['the quadratic assignment problem (qap) has r...,...,,,,,,,,,,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7749,"['nadal, cathe desiree']",['self-funded'],"['cavite', 'cavite, philippines', 'ph', 'phili...",,"['nadal, cathe desire', 'del rosario, tovie cl...",['2022-10-11t14:40:23z'],['2022-10-11t14:40:23z'],"['nadal, cathe desire; del rosario, tovie clar...","['https://hdl.handle.net/10283/4758', 'https:/...","['""capturing the dynamic character of the phil...",...,,,,,,,,,,NaN
7750,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['sefton', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t15:43:57z'],['2023-05-17t15:43:57z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/7742', 'https:/...","[""multi-storey block details: one 16-storey bl...",...,,,,,,,,,,NaN
7751,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['camden london', 'united kingdom']",['start=1988; end=1988; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t13:34:44z'],['2023-05-17t13:34:44z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/5355', 'https:/...","[""multi-storey block details: cs 1: eleven 6-s...",...,,,,,,,,,,NaN
7752,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['salford city', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t15:41:06z'],['2023-05-17t15:41:06z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/7619', 'https:/...","[""multi-storey block details: one 20-storey bl...",...,,,,,,,,,,NaN


In [30]:
#nltk.download('punkt_tab')

In [31]:
df2 = df_datashare_collapse.reset_index(drop=True)
col = "dc.description.abstract"

for i, row in df2.iterrows():
    print(f"Processing row {i+1}/{len(df2)}")

    value_to_clean = row[col]
    if pd.notna(value_to_clean) and value_to_clean != "":
        df2.at[i, "dc.description.abstract_clean"] = clean_text(value_to_clean)

df_datashare_collapse = df2

Processing row 1/7754
Processing row 2/7754
Processing row 3/7754
Processing row 4/7754
Processing row 5/7754
Processing row 6/7754
Processing row 7/7754
Processing row 8/7754
Processing row 9/7754
Processing row 10/7754
Processing row 11/7754
Processing row 12/7754
Processing row 13/7754
Processing row 14/7754
Processing row 15/7754
Processing row 16/7754
Processing row 17/7754
Processing row 18/7754
Processing row 19/7754
Processing row 20/7754
Processing row 21/7754
Processing row 22/7754
Processing row 23/7754
Processing row 24/7754
Processing row 25/7754
Processing row 26/7754
Processing row 27/7754
Processing row 28/7754
Processing row 29/7754
Processing row 30/7754
Processing row 31/7754
Processing row 32/7754
Processing row 33/7754
Processing row 34/7754
Processing row 35/7754
Processing row 36/7754
Processing row 37/7754
Processing row 38/7754
Processing row 39/7754
Processing row 40/7754
Processing row 41/7754
Processing row 42/7754
Processing row 43/7754
Processing row 44/77

In [32]:
df_datashare_collapse

,dc.contributor,dc.contributor.other,dc.coverage.spatial,dc.coverage.temporal,dc.creator,dc.date.accessioned,dc.date.available,dc.identifier.citation,dc.identifier.uri,dc.description.abstract,...,dc.description,dc.description.sponsorship,dc.relation.ispartofseries,ds.withdrawn.showtombstone,dc.relation.isbasedon,dcterms.rights,dc.relation.ispartof,dcterms.isReplacedBy,dcterms.publisher,dc.description.abstract_clean
0,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['wolverhampton city', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t16:00:22z'],['2023-05-17t16:00:22z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/8181', 'https:/...","[""multi-storey block details: two 8-storey blo...",...,,,,,,,,,,"[block, details, two, blocks, containing, dwel..."
1,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['enfield london', 'united kingdom']",['start=1988; end=1988; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t13:55:10z'],['2023-05-17t13:55:10z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/5559', 'https:/...","[""multi-storey block details: four 12-storey b...",...,,,,,,,,,,"[block, details, four, blocks, containing, dwe..."
2,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['ab11 8tw; ab11 8tx; ab11 8ty', 'aberdeen', '...",['start=1983; end=1983; scheme=w3c-dtf'],"['glendinning, miles']",['2019-11-14t09:55:05z'],['2019-11-14t09:55:05z'],"['glendinning, miles. (2019). sc_1141943.png, ...","['https://hdl.handle.net/10283/3445', 'https:/...","[""multi-storey block details: three 14-storey ...",...,,,,,,,,,,"[block, details, three, blocks, containing, dw..."
3,"['kim, jinho']",['university of edinburgh'],,,"['kim, jinho']",['2017-06-14t11:00:41z'],['2017-06-14t11:00:41z'],"['kim, jinho. (2017). source codes of the netw...","['https://hdl.handle.net/10283/2737', 'https:/...",['this data set contains the source codes for ...,...,,,,,,,,,,"[data, set, contains, source, codes, simulatio..."
4,"['anjos, miguel']",['university of edinburgh'],,,"['burkard, re', 'çela, e', 'karisch, se', 'ren...",['2022-03-31t16:59:08z'],['2022-03-31t16:59:08z'],"['burkard, re; çela, e; karisch, se; rendl, f;...","['https://hdl.handle.net/10283/4390', 'https:/...",['the quadratic assignment problem (qap) has r...,...,,,,,,,,,,"[quadratic, assignment, problem, qap, remained..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7749,"['nadal, cathe desiree']",['self-funded'],"['cavite', 'cavite, philippines', 'ph', 'phili...",,"['nadal, cathe desire', 'del rosario, tovie cl...",['2022-10-11t14:40:23z'],['2022-10-11t14:40:23z'],"['nadal, cathe desire; del rosario, tovie clar...","['https://hdl.handle.net/10283/4758', 'https:/...","['""capturing the dynamic character of the phil...",...,,,,,,,,,,"[capturing, dynamic, character, philippine, la..."
7750,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['sefton', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t15:43:57z'],['2023-05-17t15:43:57z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/7742', 'https:/...","[""multi-storey block details: one 16-storey bl...",...,,,,,,,,,,"[block, details, one, block, containing, dwell..."
7751,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['camden london', 'united kingdom']",['start=1988; end=1988; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t13:34:44z'],['2023-05-17t13:34:44z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/5355', 'https:/...","[""multi-storey block details: cs 1: eleven 6-s...",...,,,,,,,,,,"[block, details, cs, eleven, blocks, containin..."
7752,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['salford city', 'united kingdom']",['start=1987; end=1

In [33]:
list_headers_stats_clean = ["dc.description.abstract_clean"]

In [34]:
df_datashare_collapse["dc.description.abstract_clean_word_count"] = pd.Series(index = df_datashare_collapse.index, dtype="object")

In [35]:
# Word count
df_datashare_collapse["dc.description.abstract_clean_word_count"] = (
    df_datashare_collapse["dc.description.abstract_clean"]
      .apply(lambda x: len(x) if isinstance(x, list) else 0)
)

In [36]:
# Creating the empty columns for lemma text and entities 
for header in list_headers_stats_clean:
    df_datashare_collapse[f"{header}_lemmatize"] = pd.Series(index = df_datashare_collapse.index, dtype="object")
    df_datashare_collapse[f"{header}_entities"]  = pd.Series(index = df_datashare_collapse.index, dtype="object")

In [37]:
def to_text(x):
    # missing
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    # already a string
    if isinstance(x, str):
        return x
    # list/tuple/set/np array of tokens
    if isinstance(x, (list, tuple, set, np.ndarray)):
        return " ".join(map(str, x))
    # fallback: convert whatever it is to string
    return str(x)


In [38]:
# Lemmatisation and entity recognition
for idx, row in df_datashare_collapse.iterrows():
    for header in list_headers_stats_clean: 
        
        text = df_datashare_collapse.at[idx, header]
        join_text = to_text(text)
        
        lemma_str = lemmatizer("" if pd.isna(join_text) else str(join_text))
        title_lemma = f"{header}_lemmatize"
        df_datashare_collapse.at[idx, title_lemma] = lemma_str

        entities = entities_recognition(lemma_str)
        title_entities = f"{header}_entities"
        df_datashare_collapse.at[idx, title_entities] = entities

In [39]:
df_datashare_collapse

,dc.contributor,dc.contributor.other,dc.coverage.spatial,dc.coverage.temporal,dc.creator,dc.date.accessioned,dc.date.available,dc.identifier.citation,dc.identifier.uri,dc.description.abstract,...,ds.withdrawn.showtombstone,dc.relation.isbasedon,dcterms.rights,dc.relation.ispartof,dcterms.isReplacedBy,dcterms.publisher,dc.description.abstract_clean,dc.description.abstract_clean_word_count,dc.description.abstract_clean_lemmatize,dc.description.abstract_clean_entities
0,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['wolverhampton city', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t16:00:22z'],['2023-05-17t16:00:22z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/8181', 'https:/...","[""multi-storey block details: two 8-storey blo...",...,,,,,,,"[block, details, two, blocks, containing, dwel...",149,block detail two block contain dwelling block ...,"[(block), (block), (block), (wulfruna), (wulfr..."
1,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['enfield london', 'united kingdom']",['start=1988; end=1988; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t13:55:10z'],['2023-05-17t13:55:10z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/5559', 'https:/...","[""multi-storey block details: four 12-storey b...",...,,,,,,,"[block, details, four, blocks, containing, dwe...",153,block detail four block contain dwelling block...,"[(block), (block), (block), (picardy, house, n..."
2,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['ab11 8tw; ab11 8tx; ab11 8ty', 'aberdeen', '...",['start=1983; end=1983; scheme=w3c-dtf'],"['glendinning, miles']",['2019-11-14t09:55:05z'],['2019-11-14t09:55:05z'],"['glendinning, miles. (2019). sc_1141943.png, ...","['https://hdl.handle.net/10283/3445', 'https:/...","[""multi-storey block details: three 14-storey ...",...,,,,,,,"[block, details, three, blocks, containing, dw...",146,block detail three block contain dwelling bloc...,"[(block, detail), (block), (block), (grampian)..."
3,"['kim, jinho']",['university of edinburgh'],,,"['kim, jinho']",['2017-06-14t11:00:41z'],['2017-06-14t11:00:41z'],"['kim, jinho. (2017). source codes of the netw...","['https://hdl.handle.net/10283/2737', 'https:/...",['this data set contains the source codes for ...,...,,,,,,,"[data, set, contains, source, codes, simulatio...",23,datum set contain source code simulation prese...,"[(code, simulation), (thompson, centralized, r..."
4,"['anjos, miguel']",['university of edinburgh'],,,"['burkard, re', 'çela, e', 'karisch, se', 'ren...",['2022-03-31t16:59:08z'],['2022-03-31t16:59:08z'],"['burkard, re; çela, e; karisch, se; rendl, f;...","['https://hdl.handle.net/10283/4390', 'https:/...",['the quadratic assignment problem (qap) has r...,...,,,,,,,"[quadratic, assignment, problem, qap, remained...",39,quadratic assignment problem qap remain one gr...,"[(quadratic, assignment, problem, qap), (combi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7749,"['nadal, cathe desiree']",['self-funded'],"['cavite', 'cavite, philippines', 'ph', 'phili...",,"['nadal, cathe desire', 'del rosario, tovie cl...",['2022-10-11t14:40:23z'],['2022-10-11t14:40:23z'],"['nadal, cathe desire; del rosario, tovie clar...","['https://hdl.handle.net/10283/4758', 'https:/...","['""capturing the dynamic character of the phil...",...,,,,,,,"[capturing, dynamic, character, philippine, la...",49,capture dynamic character philippine landscape...,"[(dynamic), (landscape), (landscape), (cavite)..."
7750,"['glendinning, miles', 'breen, kat']",['heritage lottery fund'],"['sefton', 'united kingdom']",['start=1987; end=1987; scheme=w3c-dtf'],"['glendinning, miles']",['2023-05-17t15:43:57z'],['2023-05-17t15:43:57z'],"['glendinning, miles. (2023). tower blocks uk:...","['https://hdl.handle.net/10283/7742', 'https:/...","[""multi-storey block d

### Saving the dataset for future work 

In [40]:
os.getcwd()

'/home/eidf128/eidf128/shared/export/juliana/export/juliana/items'

In [41]:
cd ..

/home/eidf128/eidf128/shared/export/juliana/export/juliana


In [42]:
df_datashare_collapse.to_csv("df_datashare_collpase_20260311.csv")

### Organize by communities

In [43]:
pwd

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [44]:
with open("items_medicine_datashare_oct2024.txt", "r", encoding="utf-8") as f:
    items_medicine = [line.strip() for line in f if line.strip()]  # strips \n and skips blank lines

items_medicine

['6565c13c-ceec-42cb-b180-fac9d0d70cd4',
 '284b0f32-35a6-4397-bd73-2e33e3a07760',
 '7d2543c5-41f2-48be-bef1-05e18565fa88',
 '221e08d6-3494-4927-aef3-e2bfb3c13274',
 '0556a87c-baad-44db-a738-b243f113cfb2',
 '328ad395-1ac4-4a4c-8d27-5b95577eabfc',
 '7c27ab9f-2c89-45a7-b02e-0c809b13ca25',
 'd6601a32-c3f1-4114-8f84-06c133579dd2',
 '74369ef7-ad36-4f28-a3ca-f109aff00012',
 'b2537efa-43a6-4827-a53c-210bafc5d997',
 '36280bf4-0599-4102-bd44-dc0dad71864b',
 '6e1770f2-4991-44b4-a80e-011877c11c2a',
 '43408586-d464-4708-886c-6643e3be2b56',
 'd57a2aff-e21b-4ac0-9bc6-82ae98849ea5',
 '40b3a371-396f-44bc-9e89-8b7ec42f5215',
 '5fba0c7d-7f28-4a49-8fc2-3a1370aa4b4a',
 '674861f3-06ba-496a-899f-235fcb69e03d',
 '91709c88-d913-45d8-8019-380027133d54',
 '917f03d6-9eae-4b3d-b65c-b3a14cb350ba',
 '7a79f454-c747-4508-aa51-7b1bd412ce87',
 '738c746b-236e-4111-9b5d-64e181620b12',
 '7e6927f7-d321-47c0-8e1f-998f8c50433a',
 'c82e227a-af6c-4933-a242-71ee96fb0491',
 '7091a3e4-4ed3-46a7-b004-f9cf557795bd',
 'be522ba9-f16e-

In [45]:
with open("items_arts_datashare_oct2024.txt", "r", encoding="utf-8") as f:
    items_arts = [line.strip() for line in f if line.strip()]  # strips \n and skips blank lines

items_arts

['6938dbb0-21f2-44ee-b8da-a1323f5c0dd3',
 '0e3d03bd-bec3-4087-a9cd-b65aeb91786b',
 '520381ab-72b6-4824-8460-e0fcc647286a',
 '498e11e4-d1c7-4449-9937-918d82024a3d',
 '6237ff2c-1854-4a8d-853b-07e10c268dc1',
 '0a2a9035-4398-4af4-9aa9-73b93f0cb234',
 'cd4fa368-7373-4fe5-97cf-f7bfbea903e2',
 'b413fc38-1407-47bf-82dd-369dfca1d0ff',
 'fa043a92-abe5-4e4f-84a8-fe1ee929fd02',
 '324547e5-5334-49f9-b5f4-c69e5f37847b',
 'cf96b0b7-2935-49ee-84ce-8cd2ae39df8a',
 '9ce07b58-9a37-4116-8d9f-3cf67045303a',
 '7ffd32c8-6de1-4e26-ad99-df38d5375882',
 'fc5bbb0c-1e4a-493a-b7ba-7681804949d8',
 '10d58017-3006-4222-ba5b-48d86393ef6a',
 '945e027d-30f9-4b4c-bc76-073836b6057a',
 'f6cf6974-ddbd-4737-ba38-513d1d35992e',
 '9a74d10d-9cc4-4307-9751-22a0c00ff14f',
 '2b861c2a-eded-4643-85cc-8b64b92027fc',
 '9d45cba9-e757-4f9d-a3bd-8aa2a50ac2c9',
 '639ec43a-0bb9-4f33-a164-21850d198eb2',
 '7897f448-8b36-4f1a-83ea-1da22598a555',
 'aa9265a1-ab5a-4400-b8bb-f9d80a869af6',
 'c4ca94c7-7cfa-4f1e-b126-a0db7261829b',
 'aedbbb28-fbe1-

In [46]:
with open("items_sciences_datashare_oct2024.txt", "r", encoding="utf-8") as f:
    items_science = [line.strip() for line in f if line.strip()]  # strips \n and skips blank lines

items_science

['33abab4d-b941-4bf1-b27f-dc2404873bed',
 'daa15aab-1579-4d53-94d3-6200afcf9225',
 'e6b41690-ae54-4fd3-b5fb-e9d004677378',
 'b7bcb32f-a26d-422a-bd94-0cb58305ab14',
 '7f76afd3-0c78-4dbe-b937-0060aff81128',
 '857329d3-f6c0-4738-8a31-9ead1702ec5e',
 '70291716-bec9-4afb-b863-9981bcda1d52',
 'e8e22326-bba8-464f-8118-c17db718b348',
 '34f058f9-811f-462e-bcd3-f8b7d9d9789f',
 '2fdc71e7-f96e-4afa-a700-b53d5aee737a',
 '9a520471-ffe7-44c4-a1e2-3401edd50a8d',
 '63e43e02-61ad-4a04-846e-685d90d1ba4e',
 '532cb51b-bc1f-4f33-953c-12075cade7b7',
 '8ac6b1e2-b56b-4cf7-8030-fb920574979f',
 '31ef29a0-9815-495e-9e3c-4f0683fc5ddf',
 'c0deb700-63af-4a26-a1bd-76222465fb04',
 'bf092a1a-bb20-4212-bcfc-6a6aed23b87e',
 '419527df-f78b-4b67-a3ff-f0d6ee4ea6d5',
 '750e30c3-1da5-4925-a529-3e8186613bfc',
 '5426c2fd-a134-44cd-b0a7-273ddbebc6ca',
 'ce42f02b-3245-4a5c-a527-aef8ce7791d2',
 '966044f2-14ba-4c50-ae69-f4f19fe32d70',
 '18530eef-aa66-4f16-b3b4-71c74b805560',
 'ccf0c907-e6d8-4d01-a7fb-8631e8a86fc2',
 '9ce217f7-7641-

In [47]:
with open("items_thematic_datashare_oct2024.txt", "r", encoding="utf-8") as f:
    items_thematic = [line.strip() for line in f if line.strip()]  # strips \n and skips blank lines

items_thematic

['d2fc1478-cfa6-4c0e-8b55-d5d42a3ef279',
 '9aa976fb-6e7e-4ac9-87c6-439efc9b75d3',
 'ebd7b9bf-63bb-4941-8b2c-c32a7c9b0276',
 '971bf000-9b4d-4b11-b6cf-51750e9e051d',
 '97835c7d-3e65-4514-85fc-78cede84925e',
 '439f0f7c-4cad-4eb4-b418-59990c3b4637',
 'b20446df-1f61-4472-9992-da5182f5a8f2',
 'e95d0a56-7884-4155-9fb1-529ddc02aa11',
 '1af232b7-d511-4cc0-9bbc-96d6fae75aee',
 '3f6952a3-13c3-4a72-a424-33cab60f1644',
 '7fb3b4c8-6847-4c68-b900-13c71e4e5a21',
 '4c69743e-9776-4d76-a395-74e0e42c0ccb',
 'd228a88a-f77d-4585-99cf-cc42265c2071',
 '1f9c0432-80e1-4528-8c19-016193e46984',
 'a9f26f49-43fe-4cb6-ba77-ed1e83512737',
 '15e82917-56ed-4282-99ef-7f211bf87f9d',
 'd10f0b0a-e3ee-4416-8420-0781815c3bd2',
 '40ad4ad7-d731-4ddb-8b5a-51350a0dd731',
 '57f7df6c-0a26-4665-a6e7-b091160e05aa',
 '7ef02c9a-8f06-449c-ae58-34b1330053fe',
 'e998be81-fd5b-496c-b352-f90407716848',
 'd9353382-628c-4085-bebd-11bdcfde39de',
 'd0bf1a5b-38b0-4c57-a550-7e09b898436f',
 '534a77f9-b84a-420a-9587-7dbf5558d752',
 '47b99bea-b844-

In [48]:
with open("items_support_datashare_oct2024.txt", "r", encoding="utf-8") as f:
    items_support = [line.strip() for line in f if line.strip()]  # strips \n and skips blank lines

items_support

['a8427d6f-ae38-42dc-8631-365e709f79b3',
 '4d9d1f69-9dda-4eb7-b3c5-4c6b3c454c2c',
 '8407e11d-0489-419d-9a8e-2176beb339a3',
 '1e8cdbeb-fad8-496d-9ced-9ddb1501ae8f',
 '37965637-dcdc-4309-a83b-893400466ac8',
 'a01d08ed-d07a-4933-baa6-c2b158e333f6',
 'c1264cd6-6bd1-4e0b-9d60-3536e861c2f3',
 'fd8cefc1-8a92-44dd-ba37-617992ffeea4',
 '02f77652-d9dd-4f96-ab04-778bce3d12ca',
 '577cea5b-4291-4b6d-b686-be0f21b1288e',
 '526e7fc5-3b40-4082-be55-4c5381bcd3ce',
 'ea31ce42-b815-42e5-8fbc-e67de98222ed',
 'd58bf0a3-a5da-4f86-bb7d-805bf6a814b4',
 '54ac5fa5-ba09-42c5-8119-5917ea229a1d',
 '0037d8c3-f792-4106-a255-2594846e9a97',
 '9a0da7a8-8324-48fa-a31f-c620b762d34c',
 'efe57d9c-cd7f-4195-87a8-ff98ff0bb9e4',
 '40961da8-2be5-400a-a7fb-a32854fdd2ac',
 '747da831-5bb6-4701-8298-7b8857668ffb',
 '9bf40b37-a9e7-428c-9bf6-9c13f288883a',
 '217d495b-0712-4824-a36e-2b11b88cc0f7',
 '58910db2-b0e1-4f02-95df-51368b775e6d',
 '61e6836b-ad45-4ac6-b825-a58064b8dec8',
 'b80afaff-b3cb-4f72-87c8-25d148ba1801',
 'e04feeb5-9d28-

In [49]:
df_datashare_collapse.columns

Index(['dc.contributor', 'dc.contributor.other', 'dc.coverage.spatial',
       'dc.coverage.temporal', 'dc.creator', 'dc.date.accessioned',
       'dc.date.available', 'dc.identifier.citation', 'dc.identifier.uri',
       'dc.description.abstract', 'dc.language.iso', 'dc.publisher',
       'dc.relation.isreferencedby', 'dc.rights', 'dc.subject',
       'dc.subject.classification', 'dc.title', 'dc.type', 'file_name',
       'collection', 'id_in_file', 'dc.relation.isversionof',
       'dc.description.tableofcontents', 'dc.source', 'dc.title.alternative',
       'dc.relation.isreplacedby', 'dc.date.updated', 'dc.date.embargo',
       'dc.relation.replaces', 'dc.source.uri', 'dc.date.issued',
       'dcterms.subject', 'dcterms.isReferencedBy', 'dc.contributor.advisor',
       'dc.relation.hasversion', 'dc.contributor.author', 'dc.description',
       'dc.description.sponsorship', 'dc.relation.ispartofseries',
       'ds.withdrawn.showtombstone', 'dc.relation.isbasedon', 'dcterms.rights',


In [50]:
df_datashare_collapse["id_in_file"]

0       49dbc9e1-eeab-411b-8510-080ceb5dccfa
1       ecfd3009-d585-4b1a-9137-6b7460379b94
2       619bb04f-94c4-4eb8-9511-b2dfeaa3412a
3       e232d2db-1621-47df-bb6d-ea69187de44f
4       9bf808da-4a87-4bfc-a885-4f53bf7531fc
                        ...                 
7749    95b48a67-d9d6-497b-92f8-eca8ba9eb13e
7750    5a681507-5f98-4b33-821a-f7a14e27e6e7
7751    f5a80a20-a794-45e8-8f67-de9031a43037
7752    163fe91b-f71a-4337-9baf-8bfe1ddbb9ae
7753    2a18550b-fdaa-47a4-94a2-5359ae0266a3
Name: id_in_file, Length: 7754, dtype: string

In [51]:
df_medicine = df_datashare_collapse[df_datashare_collapse["id_in_file"].isin(items_medicine)].copy()

In [52]:
df_arts = df_datashare_collapse[df_datashare_collapse["id_in_file"].isin(items_arts)].copy()

In [53]:
df_science = df_datashare_collapse[df_datashare_collapse["id_in_file"].isin(items_science)].copy()

In [54]:
df_support = df_datashare_collapse[df_datashare_collapse["id_in_file"].isin(items_support)].copy()

In [55]:
df_thematic = df_datashare_collapse[df_datashare_collapse["id_in_file"].isin(items_thematic)].copy()

#### Saving the files

In [56]:
df_medicine.to_csv("df_medicine_datashare_collpase_20260401.csv")

In [57]:
df_arts.to_csv("df_arts_datashare_collpase_20260401.csv")

In [58]:
df_science.to_csv("df_science_datashare_collpase_20260401.csv")

In [59]:
df_support.to_csv("df_support_datashare_collpase_20260401.csv")

In [60]:
df_thematic.to_csv("df_thematic_collpase_20260401.csv")